# 07 Inference Engine Walkthrough

This notebook explains how nanochat turns a chat prompt into a generated assistant response after SFT.

The important runtime path is:

```text
chat messages
    -> chat_web.py builds prompt token ids
    -> Engine.generate(...) prefills the prompt into a KV cache
    -> sample one next token
    -> decode one token at a time using the KV cache
    -> stop when <|assistant_end|> is generated
```

This goes deeper than `04_eval_and_generation_walkthrough.ipynb`. Notebook 04 explains why generation and evaluation matter. This notebook follows the internal inference engine carefully.

The cells use nanochat's real tokenizer and real `Engine` helper classes when available, but intentionally avoid loading the large SFT model weights by default.

In [1]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not ((repo_root / "nanochat").exists() and (repo_root / "pyproject.toml").exists()):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "nanochat").exists() and (candidate / "pyproject.toml").exists():
            repo_root = candidate
            break

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

missing = []
for package in ["torch", "rustbpe", "tiktoken", "tokenizers"]:
    try:
        __import__(package)
    except Exception:
        missing.append(package)

if missing:
    raise RuntimeError(
        "Missing dependencies: " + ", ".join(missing) + "\n"
        "From the repo root run:\n"
        "  uv sync --extra cpu --group dev\n"
        "or, on CUDA machines:\n"
        "  uv sync --extra gpu --group dev\n"
        "Then reopen this notebook with the nanochat (.venv) kernel."
    )

import torch

from nanochat.common import get_base_dir, COMPUTE_DTYPE, COMPUTE_DTYPE_REASON
from nanochat.engine import KVCache, RowState, sample_next_token, use_calculator
from nanochat.tokenizer import RustBPETokenizer

artifact_root = Path(os.environ.get("NANOCHAT_ARTIFACTS_DIR", repo_root / "nanochat-artifacts")).expanduser()

def find_tokenizer_dir():
    candidates = [
        artifact_root / "tokenizer",
        Path(get_base_dir()) / "tokenizer",
    ]
    for candidate in candidates:
        if (candidate / "tokenizer.pkl").exists():
            return candidate
    return None

tokenizer_dir = find_tokenizer_dir()
tokenizer = RustBPETokenizer.from_directory(tokenizer_dir) if tokenizer_dir else None

print(f"repo root:      {repo_root}")
print(f"artifact root:  {artifact_root} (exists={artifact_root.exists()})")
print(f"base dir:       {get_base_dir()}")
print(f"tokenizer dir:  {tokenizer_dir}")
print(f"compute dtype:  {COMPUTE_DTYPE} ({COMPUTE_DTYPE_REASON})")

repo root:      /Users/eugene/Developer/nanochat
artifact root:  /Users/eugene/Developer/nanochat/nanochat-artifacts (exists=True)
base dir:       /Users/eugene/.cache/nanochat
tokenizer dir:  /Users/eugene/Developer/nanochat/nanochat-artifacts/tokenizer
compute dtype:  torch.float32 (auto-detected: no CUDA (CPU/MPS))


## Step 1. `chat_web.py` Builds The Inference Prompt

The web UI receives normal chat messages such as:

```python
[{"role": "user", "content": "Tell me a short joke."}]
```

Before calling the engine, [`scripts/chat_web.py`](../scripts/chat_web.py#L325-L341) converts those messages into token ids and appends `<|assistant_start|>`:

```text
<|bos|>
<|user_start|> Tell me a short joke. <|user_end|>
<|assistant_start|>
```

The final marker means: *the assistant should continue from here*.

The engine works with token ids, not raw text. Tokenization happens before the engine receives the prompt.

In [2]:
def build_chat_prompt(tokenizer, messages):
    """Small direct mirror of scripts/chat_web.py lines 325-341."""
    bos = tokenizer.get_bos_token_id()
    user_start = tokenizer.encode_special("<|user_start|>")
    user_end = tokenizer.encode_special("<|user_end|>")
    assistant_start = tokenizer.encode_special("<|assistant_start|>")
    assistant_end = tokenizer.encode_special("<|assistant_end|>")

    conversation_tokens = [bos]
    for message in messages:
        if message["role"] == "user":
            conversation_tokens.append(user_start)
            conversation_tokens.extend(tokenizer.encode(message["content"]))
            conversation_tokens.append(user_end)
        elif message["role"] == "assistant":
            conversation_tokens.append(assistant_start)
            conversation_tokens.extend(tokenizer.encode(message["content"]))
            conversation_tokens.append(assistant_end)

    conversation_tokens.append(assistant_start)
    return conversation_tokens

messages = [{"role": "user", "content": "Tell me a short joke."}]

if tokenizer is None:
    prompt_ids = None
    print("No trained tokenizer found. The remaining conceptual cells still work.")
else:
    prompt_ids = build_chat_prompt(tokenizer, messages)
    print("Prompt token ids:")
    print(prompt_ids)
    print()
    print("Decoded prompt passed to Engine.generate(...):")
    print(tokenizer.decode(prompt_ids))


Prompt token ids:
[32759, 32760, 27335, 522, 257, 1764, 17824, 46, 32761, 32762]

Decoded prompt passed to Engine.generate(...):
<|bos|><|user_start|>Tell me a short joke.<|user_end|><|assistant_start|>


## Step 2. GPT Logits Become The First Generated Token

Step 1 built the prompt token ids. The engine now sends those ids into GPT.

GPT does not directly return a word or sentence. It returns **logits**: one score for every possible next token in the tokenizer vocabulary.

For the prompt:

```text
<|bos|><|user_start|>Tell me a short joke.<|user_end|><|assistant_start|>
```

the model might conceptually produce scores like:

| Candidate next token | Logit |
|---|---:|
| `Why` | `3.2` |
| `Because` | `1.8` |
| `A` | `1.1` |
| `The` | `0.7` |
| <code>&lt;&#124;assistant_end&#124;&gt;</code> | `-1.0` |

[`sample_next_token(...)`](../nanochat/engine.py#L141-L159) turns those logits into one selected next-token id.

- `temperature=0`: greedy decoding, always choose the highest logit.
- lower positive temperature: prefer likely tokens more strongly.
- higher temperature: allow more variety.
- `top_k=K`: only sample among the `K` strongest candidates.

The real prefill path is in [`Engine.generate(...)`](../nanochat/engine.py#L198-L207):

```python
ids = torch.tensor([tokens], dtype=torch.long, device=device)
logits = self.model.forward(ids, kv_cache=kv_cache_prefill)
logits = logits[:, -1, :].expand(num_samples, -1)
```

GPT calculates logits at every prompt position, but the engine keeps only the last position because it needs the token that comes after `<|assistant_start|>`.

In [3]:
toy_vocab = ["Why", "Because", "A", "The", "<|assistant_end|>"]
toy_logits = torch.tensor([[3.2, 1.8, 1.1, 0.7, -1.0]])

probs = torch.softmax(toy_logits, dim=-1)[0]
print(f"{'candidate token':<22} {'logit':>8} {'probability':>14}")
print("-" * 48)
for token, logit, probability in zip(toy_vocab, toy_logits[0].tolist(), probs.tolist()):
    print(f"{token:<22} {logit:>8.2f} {probability:>14.3f}")

rng = torch.Generator(device="cpu")
rng.manual_seed(42)
greedy_id = int(sample_next_token(toy_logits, rng, temperature=0.0)[0, 0])
print()
print("temperature=0 selects:", repr(toy_vocab[greedy_id]))

def draw_tokens(temperature, top_k=None, draws=8):
    rng = torch.Generator(device="cpu")
    rng.manual_seed(42)
    selected = []
    for _ in range(draws):
        token_id = int(sample_next_token(toy_logits, rng, temperature=temperature, top_k=top_k)[0, 0])
        selected.append(toy_vocab[token_id])
    return selected

print("temperature=0.7 samples:       ", draw_tokens(temperature=0.7))
print("temperature=1.5 samples:       ", draw_tokens(temperature=1.5))
print("temperature=1.5 top_k=2 samples:", draw_tokens(temperature=1.5, top_k=2))


candidate token           logit    probability
------------------------------------------------
Why                        3.20          0.682
Because                    1.80          0.168
A                          1.10          0.084
The                        0.70          0.056
<|assistant_end|>         -1.00          0.010

temperature=0 selects: 'Why'
temperature=0.7 samples:        ['Why', 'Why', 'Why', 'Because', 'Why', 'Why', 'Why', 'A']
temperature=1.5 samples:        ['Why', 'The', 'Why', 'Because', 'Why', 'Why', 'Why', 'A']
temperature=1.5 top_k=2 samples: ['Why', 'Why', 'Why', 'Why', 'Why', 'Why', 'Why', 'Why']


## Step 3. `Engine.generate(...)` Yields One Token At A Time

Step 2 selected the first next token. The engine returns that selected token to its caller, then continues decoding.

[`Engine.generate(...)`](../nanochat/engine.py#L167-L276) is a Python generator:

```python
for token_column, token_masks in engine.generate(prompt_ids, ...):
    ...
```

Each loop iteration yields:

```text
token_column: one newly generated token id per sample row
token_masks:  metadata describing where each returned token came from
```

For an ordinary one-sample chat response, the stream is conceptually:

```text
iteration 1 -> token_column=['Why']
iteration 2 -> token_column=[' did']
iteration 3 -> token_column=[' the']
...
```

If `num_samples=3`, each `token_column` contains one newly selected token for each of three alternative responses.

### What `token_masks` Are For

For normal text generated by the model, `token_mask=1`.

`token_masks` become interesting when the model requests calculator use. They do **not** trigger calculator execution. Special tokens trigger the calculator state machine:

```text
<|python_start|> 3+4 <|python_end|>
```

When the engine sees `<|python_end|>`, it evaluates the expression and automatically inserts:

```text
<|output_start|> 7 <|output_end|>
```

`token_masks` record where each token came from:

```text
mask=1 -> model sampled this token from logits
mask=0 -> engine forced this token into the stream
```

| Token | Source | `token_mask` |
|---|---|---:|
| <code>&lt;&#124;python_start&#124;&gt;</code> | generated by model | `1` |
| `3` | generated by model | `1` |
| `+` | generated by model | `1` |
| `4` | generated by model | `1` |
| <code>&lt;&#124;python_end&#124;&gt;</code> | generated by model | `1` |
| <code>&lt;&#124;output_start&#124;&gt;</code> | inserted by engine | `0` |
| `7` | inserted by calculator | `0` |
| <code>&lt;&#124;output_end&#124;&gt;</code> | inserted by engine | `0` |

For ordinary chat inference, treat `token_masks` as metadata. They do not control generation.

They become important later during reinforcement learning. [`scripts/chat_rl.py`](../scripts/chat_rl.py#L131-L138) converts targets with mask `0` into `-1`, so RL trains on choices made by the model but does not train the model to imitate calculator output supplied by the engine.

In [4]:
ordinary_stream = [
    ("Why", 1),
    (" did", 1),
    (" the", 1),
    (" chicken", 1),
    (" cross", 1),
    (" the", 1),
    (" road", 1),
    ("?", 1),
]

print("Ordinary one-sample token stream returned by Engine.generate(...):")
print(f"{'iteration':>9}  {'token_column':<22} {'token_masks'}")
print("-" * 54)
for iteration, (token, mask) in enumerate(ordinary_stream, 1):
    print(f"{iteration:>9}  {str([token]):<22} {[mask]}")

print()
print("Calculator token_masks example:")
print(f"{'token':<24} {'source':<28} {'mask'}")
print("-" * 58)
calculator_stream = [
    ("<|python_start|>", "generated by model", 1),
    ("3", "generated by model", 1),
    ("+", "generated by model", 1),
    ("4", "generated by model", 1),
    ("<|python_end|>", "generated by model", 1),
    ("<|output_start|>", "inserted by engine", 0),
    ("7", "inserted by calculator", 0),
    ("<|output_end|>", "inserted by engine", 0),
]
for token, source, mask in calculator_stream:
    print(f"{token:<24} {source:<28} {mask}")

print()
print("For normal inference these masks are metadata. Later, RL uses mask=0 to ignore forced tokens in loss.")


Ordinary one-sample token stream returned by Engine.generate(...):
iteration  token_column           token_masks
------------------------------------------------------
        1  ['Why']                [1]
        2  [' did']               [1]
        3  [' the']               [1]
        4  [' chicken']           [1]
        5  [' cross']             [1]
        6  [' the']               [1]
        7  [' road']              [1]
        8  ['?']                  [1]

Calculator token_masks example:
token                    source                       mask
----------------------------------------------------------
<|python_start|>         generated by model           1
3                        generated by model           1
+                        generated by model           1
4                        generated by model           1
<|python_end|>           generated by model           1
<|output_start|>         inserted by engine           0
7                        inserted by calc

## Step 4. Inference Has A Prefill Phase And A Decode Phase

During training, a whole row is processed together with causal attention because the correct continuation is already known.

During inference, the answer does not exist yet. The engine must generate one new token at a time. To avoid recomputing the prompt repeatedly, it uses two phases:

```text
PREFILL
  process the full prompt once
  cache the attention keys and values for prompt positions
  obtain logits for the first answer token

DECODE LOOP
  sample one next token
  process only that one new token
  append its keys and values to the cache
  obtain logits for the following token
  repeat
```

For the running joke prompt, imagine that Step 2 selected `Why` as the first answer token:

```text
prompt:
  <|bos|><|user_start|>Tell me a short joke.<|user_end|><|assistant_start|>

prefill:
  model input = full prompt
  cache       = keys/values for every prompt token
  output      = logits used to choose 'Why'

decode step 1:
  model input = only 'Why'
  cache       = previous prompt cache + new keys/values for 'Why'
  output      = logits used to choose ' did'

decode step 2:
  model input = only ' did'
  cache       = previous cache + new keys/values for ' did'
  output      = logits used to choose ' the'
```

The important part is that the model receives the whole prompt exactly once. After that, each decode call receives only the newest token because earlier context is already represented by cached keys and values.

The KV cache is an inference optimization. It does not change what the model predicts.

Causal attention still applies in both phases. A token can use earlier cached context, but it cannot see future tokens. Nanochat can also use sliding-window attention in some layers, which limits how far backward a token can look without changing that causal rule.

In [5]:
prompt_length = len(prompt_ids) if prompt_ids is not None else 10
generated_tokens = ["Why", " did", " the"]

print("Concrete prefill/decode timeline:")
print(f"{'call':<14} {'model input':<24} {'model input shape':<20} {'cache positions after call'}")
print("-" * 90)
print(f"{'prefill':<14} {'full chat prompt':<24} {str((1, prompt_length)):<20} 0..{prompt_length - 1}")
for step, token in enumerate(generated_tokens, 1):
    last_position = prompt_length + step - 1
    print(f"{'decode ' + str(step):<14} {repr(token):<24} {str((1, 1)):<20} 0..{last_position}")

print()
print("During decode, cached context grows while each new model input remains one token long.")


Concrete prefill/decode timeline:
call           model input              model input shape    cache positions after call
------------------------------------------------------------------------------------------
prefill        full chat prompt         (1, 10)              0..9
decode 1       'Why'                    (1, 1)               0..10
decode 2       ' did'                   (1, 1)               0..11
decode 3       ' the'                   (1, 1)               0..12

During decode, cached context grows while each new model input remains one token long.


## Step 5. What The KV Cache Stores

Attention compares a new query against keys from earlier positions and combines earlier values.

Without a KV cache, every generated token would recompute keys and values for the full previous prompt again.

With a KV cache, nanochat stores earlier keys and values using this shape from [`KVCache`](../nanochat/engine.py#L82-L138):

```text
(num_layers, batch_size, max_sequence_length, num_kv_heads, head_dimension)
```

The cache uses `n_kv_head`, not necessarily `n_head`. This is where GQA can reduce inference-memory cost.

In [6]:
# Tiny real KVCache objects: small enough to inspect in a notebook.
prefill_cache = KVCache(
    batch_size=1,
    num_heads=2,
    seq_len=3,
    head_dim=4,
    num_layers=2,
    device="cpu",
    dtype=torch.float32,
)

# A real model forward pass normally writes these values. Markers keep the demo lightweight.
prefill_cache.k_cache.fill_(11)
prefill_cache.v_cache.fill_(22)
prefill_cache.advance(3)

decode_cache = KVCache(
    batch_size=3,
    num_heads=2,
    seq_len=8,
    head_dim=4,
    num_layers=2,
    device="cpu",
    dtype=torch.float32,
)
decode_cache.prefill(prefill_cache)

print("Prefill cache K shape: ", tuple(prefill_cache.k_cache.shape))
print("Decode cache K shape:  ", tuple(decode_cache.k_cache.shape))
print("Decode sequence lengths:", decode_cache.cache_seqlens.tolist())
print("Prompt cache cloned to all 3 sample rows:", bool(torch.all(decode_cache.k_cache[:, :, :3] == 11)))
print()
print("The real Engine uses this copy so one prompt prefill can branch into multiple sampled answers.")


Prefill cache K shape:  (2, 1, 3, 2, 4)
Decode cache K shape:   (2, 3, 8, 2, 4)
Decode sequence lengths: [3, 3, 3]
Prompt cache cloned to all 3 sample rows: True

The real Engine uses this copy so one prompt prefill can branch into multiple sampled answers.


## Step 6. Follow The Real `Engine.generate(...)` Loop

The important lines in [`nanochat/engine.py`](../nanochat/engine.py#L192-L276) are:

```python
# 1. Prefill: run the whole prompt once.
ids = torch.tensor([tokens], dtype=torch.long, device=device)
logits = self.model.forward(ids, kv_cache=kv_cache_prefill)
logits = logits[:, -1, :].expand(num_samples, -1)

# 2. Copy prompt cache so alternative samples can branch from it.
kv_cache_decode.prefill(kv_cache_prefill)

# 3. Repeated decode step: choose one token and process only that token.
next_ids = sample_next_token(logits, rng, temperature, top_k)
ids = torch.tensor(token_column, dtype=torch.long, device=device).unsqueeze(1)
logits = self.model.forward(ids, kv_cache=kv_cache_decode)[:, -1, :]
```

The key shape change is:

```text
prefill model input: (1, prompt_length)
decode model input:  (num_samples, 1)
```

During decode, previous context lives in the KV cache. The engine only sends the newest token through the model.

In [7]:
prompt_length = len(prompt_ids) if prompt_ids is not None else 12
num_samples = 3

print(f"Prompt length: {prompt_length}")
print()
print(f"{'phase':<12} {'model input shape':<22} what is processed")
print("-" * 72)
print(f"{'prefill':<12} {str((1, prompt_length)):<22} full prompt, exactly once")
for step in range(1, 5):
    print(f"{'decode ' + str(step):<12} {str((num_samples, 1)):<22} one newest token per sample row")


Prompt length: 10

phase        model input shape      what is processed
------------------------------------------------------------------------
prefill      (1, 10)                full prompt, exactly once
decode 1     (3, 1)                 one newest token per sample row
decode 2     (3, 1)                 one newest token per sample row
decode 3     (3, 1)                 one newest token per sample row
decode 4     (3, 1)                 one newest token per sample row


## Step 7. GPT Writes New Keys And Values Into The Cache

Inside [`nanochat/gpt.py`](../nanochat/gpt.py#L106-L121), attention uses a different path when a KV cache is present:

```python
if kv_cache is None:
    # Training: causal attention over the available row.
    y = flash_attn.flash_attn_func(q, k, v, causal=True, ...)
else:
    # Inference: insert new K/V into the cache and attend over cached context.
    k_cache, v_cache = kv_cache.get_layer_cache(self.layer_idx)
    y = flash_attn.flash_attn_with_kvcache(
        q, k_cache, v_cache, k=k, v=v, ...
    )
```

The SDPA fallback implementation writes new keys and values in-place in [`nanochat/flash_attention.py`](../nanochat/flash_attention.py#L156-L184).

The cache also tracks the current sequence position. [`nanochat/gpt.py`](../nanochat/gpt.py#L424-L425) uses that offset to select the correct RoPE rotation for newly decoded tokens:

```python
T0 = 0 if kv_cache is None else kv_cache.get_pos()
cos_sin = self.cos[:, T0:T0+T], self.sin[:, T0:T0+T]
```

Nanochat also stores the previous embedding in the cache for its smear feature. That lets one-token decoding mix in the previous-token embedding without recomputing the full sequence.

## Step 8. Generation Stops On `<|assistant_end|>`

SFT trained the model to emit `<|assistant_end|>` after its answer.

During inference, [`Engine.generate(...)`](../nanochat/engine.py#L249-L250) marks that sample row as complete:

```python
if next_token == assistant_end or next_token == bos:
    state.completed = True
```

The outer loop also stops if `max_tokens` is reached. This protects the application if the model never emits its stop marker.

In [8]:
if tokenizer is None:
    print("No trained tokenizer found, skipping stop-token demo.")
else:
    assistant_end = tokenizer.encode_special("<|assistant_end|>")
    bos = tokenizer.get_bos_token_id()
    simulated_tokens = [*tokenizer.encode("Why did the chicken cross the road?"), assistant_end, *tokenizer.encode(" ignored")]
    state = RowState()

    print(f"{'generated token':<28} {'completed?'}")
    print("-" * 44)
    for token_id in simulated_tokens:
        state.current_tokens.append(token_id)
        if token_id == assistant_end or token_id == bos:
            state.completed = True
        print(f"{tokenizer.decode([token_id])!r:<28} {state.completed}")
        if state.completed:
            break

    print()
    print("Anything after <|assistant_end|> is not generated for this response.")


generated token              completed?
--------------------------------------------
'Why'                        False
' did'                       False
' the'                       False
' chicken'                   False
' cross'                     False
' the'                       False
' road'                      False
'?'                          False
'<|assistant_end|>'          True

Anything after <|assistant_end|> is not generated for this response.


## Step 9. Streaming And Batch Helpers Wrap The Same Engine

`Engine.generate(...)` yields one token column at a time. That is useful for streaming a chat response immediately.

[`scripts/chat_web.py`](../scripts/chat_web.py#L255-L303) decodes accumulated tokens and streams newly completed UTF-8 text to the browser.

[`Engine.generate_batch(...)`](../nanochat/engine.py#L278-L301) is a convenience wrapper for callers that want complete sequences instead. It intentionally does not include terminal `<|assistant_end|>` or `<|bos|>` tokens in returned results.

So:

```text
Engine.generate(...)       -> token-by-token stream, includes stop marker
Engine.generate_batch(...) -> accumulated result, strips terminal marker
```

## Step 10. Calculator Tool Results Are Forced Into The Stream

GSM8K and SpellingBee SFT examples teach the assistant to emit Python tool-call markers:

```text
<|python_start|> 3+4 <|python_end|>
```

The engine watches for those markers in [`nanochat/engine.py`](../nanochat/engine.py#L252-L268):

```text
model emits python_start
    -> engine records expression tokens
model emits python_end
    -> engine evaluates expression with use_calculator(...)
    -> engine queues output_start, result tokens, output_end
    -> queued tokens are forced into the sequence
```

The forced output tokens have `token_mask=0`: they came from the calculator, not from model sampling.

This mirrors SFT: Python outputs were visible as context but excluded from loss because the calculator supplies them at inference time.

In [9]:
expressions = [
    "3+4",
    "12/3",
    "'strawberry'.count('r')",
    "2**10",       # rejected intentionally
    "open('x')",   # rejected intentionally
]

print(f"{'expression':<30} calculator result")
print("-" * 52)
for expression in expressions:
    print(f"{expression:<30} {use_calculator(expression)!r}")

if tokenizer is not None:
    output_start = tokenizer.encode_special("<|output_start|>")
    output_end = tokenizer.encode_special("<|output_end|>")
    forced_tokens = [output_start, *tokenizer.encode("7"), output_end]
    print()
    print("Forced stream after the model emits <|python_start|>3+4<|python_end|>:")
    print(tokenizer.decode(forced_tokens))


expression                     calculator result
----------------------------------------------------
3+4                            7
12/3                           4.0
'strawberry'.count('r')        3
2**10                          None
open('x')                      None

Forced stream after the model emits <|python_start|>3+4<|python_end|>:
<|output_start|>7<|output_end|>


## Step 11. Run Real Chat Intentionally

The notebook avoids loading your large `d24` SFT checkpoint automatically. For real chat, use the CLI or web app intentionally.

CLI:

```bash
cd /path/to/nanochat
source .venv/bin/activate
export NANOCHAT_BASE_DIR=/path/to/nanochat-artifacts
python -m scripts.chat_cli --source=sft
```

Web UI:

```bash
cd /path/to/nanochat
source .venv/bin/activate
export NANOCHAT_BASE_DIR=/path/to/nanochat-artifacts
python -m scripts.chat_web --host 0.0.0.0 --port 8000
```

`scripts.chat_cli.py` and `scripts.chat_web.py` both assemble chat tokens and delegate generation to the same `Engine`.

## Mental Model

```text
1. chat_web.py wraps user text with chat special tokens
2. Engine receives a list of token ids ending in <|assistant_start|>
3. prefill runs the whole prompt once and fills the KV cache
4. sampling picks one next token from logits
5. decode runs only that newest token while reusing cached context
6. repeat until <|assistant_end|> or max_tokens
7. if the model asks for Python, Engine injects calculator output as forced context
```

The central inference optimization is simple:

```text
do not recompute old attention keys and values;
cache them and process only the new token.
```